# Chapter 6: Gaussian Distributions

<a href="../lite/lab/index.html?path=ch06_gaussian_distributions.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.stats import norm, multivariate_normal

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Helper: draw a confidence ellipse from a 2x2 covariance matrix ──────────
def cov_ellipse(ax, mu, cov, n_std=1.0, **kwargs):
    """Draw an n_std confidence ellipse for a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mu, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)
    return ell

print('Imports and helpers ready.')

```{admonition} What you will build
:class: tip

- Sample from 1D and multivariate Gaussians and verify samples match the analytical PDF
- Draw confidence ellipses that visualize robot position uncertainty
- Use the Cholesky decomposition to generate correlated samples (the technique used inside every Kalman filter)
- Implement rejection sampling and importance sampling from scratch
- Transform Gaussians through linear functions and see how ellipses stretch and rotate

**Real world application:** Gaussians model sensor noise, position uncertainty, and state beliefs in Kalman filters, EKF SLAM, and graph optimization. Sampling from them powers particle filters and Monte Carlo simulation.
```

```{admonition} Libraries and tools used in practice
:class: note

| Library / Tool | What it does |
|---|---|
| **scipy.stats.multivariate_normal** | Multivariate Gaussian PDF, sampling, and log likelihood |
| **numpy.random.multivariate_normal** | Fast sampling from multivariate Gaussians |
| **numpy.linalg.cholesky** | Cholesky decomposition for correlated sampling |
| **matplotlib.patches.Ellipse** | Drawing confidence ellipses from covariance matrices |
```

## 6.1 The 1D Gaussian

The Gaussian (or **normal**) distribution is the most important distribution in robotics.
A 1D Gaussian is fully described by two parameters:

- **Mean** $\mu$ : the center of the bell curve, our best estimate
- **Standard deviation** $\sigma$ : the spread, our uncertainty (variance is $\sigma^2$)

The probability density function (PDF) is:

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left\{-\frac{(x-\mu)^2}{2\sigma^2}\right\}$$

Physically, when a robot reports "I am at position 5.0 m with $\sigma = 0.3$ m," it means the true position is very likely within 5.0 $\pm$ 0.9 m (three sigma).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
mu    = 0.0    # mean of the Gaussian        (try -3, 0, 2, 5)
sigma = 1.5    # standard deviation           (try 0.5, 1, 2, 4)
# ─────────────────────────────────────────────────────────────────────────────

x = np.linspace(mu - 5*sigma, mu + 5*sigma, 600)
pdf = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, pdf, color='steelblue', linewidth=2.5, label='Gaussian PDF')
ax.axvline(mu, color='tomato', linestyle='--', linewidth=1.5, label=f'mean $\\mu$ = {mu:.1f}')

# Shade the 68-95-99.7 regions
for k, alpha, lbl in [(3, 0.08, '99.7% (3$\\sigma$)'),
                       (2, 0.12, '95.4% (2$\\sigma$)'),
                       (1, 0.20, '68.3% (1$\\sigma$)')]:
    mask = (x >= mu - k*sigma) & (x <= mu + k*sigma)
    ax.fill_between(x[mask], pdf[mask], alpha=alpha, color='steelblue', label=lbl)

ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title(f'1D Gaussian: $\\mu={mu:.1f}$, $\\sigma={sigma:.1f}$')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

### The 68 / 95 / 99.7 rule

The shaded regions above illustrate a fundamental property of the Gaussian:

| Region | Probability |
|--------|------------|
| $\mu \pm 1\sigma$ | 68.3% |
| $\mu \pm 2\sigma$ | 95.4% |
| $\mu \pm 3\sigma$ | 99.7% |

In robotics, a "$3\sigma$ bound" is the practical limit of plausible values. Anything outside three sigma is treated as an outlier or a sensor fault.

### A robot on a corridor

Below, a robot sits in a 1D corridor. Its belief about its own position is a Gaussian. A small $\sigma$ means high confidence; a large $\sigma$ means the robot is unsure.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
robot_pos   = 5.0    # believed position on the corridor (meters)
robot_sigma = 0.8    # position uncertainty (meters)
# ─────────────────────────────────────────────────────────────────────────────

x = np.linspace(0, 10, 500)
pdf = norm.pdf(x, robot_pos, robot_sigma)

fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(x, pdf, alpha=0.3, color='steelblue')
ax.plot(x, pdf, color='steelblue', linewidth=2)
ax.plot(robot_pos, 0, 's', color='tomato', markersize=14, zorder=5, label='Robot')
ax.annotate('Robot', (robot_pos, 0), textcoords='offset points',
            xytext=(12, 8), fontsize=11, color='tomato', fontweight='bold')

# Draw corridor walls
ax.axhline(0, color='gray', linewidth=2)
ax.set_xlim(0, 10); ax.set_ylim(-0.05 * pdf.max(), pdf.max() * 1.2)
ax.set_xlabel('Position along corridor (m)')
ax.set_ylabel('Belief  p(x)')
ax.set_title(f'Robot position belief: $\\mu={robot_pos}$ m, $\\sigma={robot_sigma}$ m')
plt.tight_layout(); plt.show()

### Sampling from a 1D Gaussian

**Sampling** means drawing random values from a distribution. Why does sampling matter?
Many distributions in robotics have no closed form solution for integrals or expectations.
Sampling lets us approximate these quantities computationally: draw $N$ samples, compute
the statistic of interest, and the law of large numbers guarantees convergence.

Below we generate 1000 samples with `np.random.normal` and overlay their histogram
on the analytical PDF. The histogram should closely track the curve.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
mu_samp     = 2.0      # mean for sampling demo
sigma_samp  = 1.0      # std deviation
n_samples   = 1000     # number of samples (try 50, 200, 5000)
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(42)
samples = rng.normal(mu_samp, sigma_samp, size=n_samples)

x = np.linspace(mu_samp - 4*sigma_samp, mu_samp + 4*sigma_samp, 400)
pdf = norm.pdf(x, mu_samp, sigma_samp)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(samples, bins=40, density=True, alpha=0.5, color='steelblue',
        edgecolor='white', label=f'Histogram ({n_samples} samples)')
ax.plot(x, pdf, 'tomato', linewidth=2.5, label='Analytical PDF')
ax.axvline(samples.mean(), color='forestgreen', linestyle='--',
           label=f'Sample mean = {samples.mean():.3f}')
ax.set_xlabel('x'); ax.set_ylabel('Density')
ax.set_title('Samples vs. Analytical PDF')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'True  mean = {mu_samp:.4f},   Sample mean = {samples.mean():.4f}')
print(f'True  std  = {sigma_samp:.4f},   Sample std  = {samples.std(ddof=1):.4f}')

**Key observations:**
- With more samples, the histogram converges to the true PDF.
- The sample mean and sample standard deviation converge to $\mu$ and $\sigma$.
- Try setting `n_samples = 50` vs. `n_samples = 5000` to see the difference.

---

## 6.2 The Multivariate Gaussian

A robot's state is rarely a single number. Even a planar robot has position $(x, y)$,
so we need a distribution over vectors. The **multivariate Gaussian** generalizes the
bell curve to $d$ dimensions:

$$p(\mathbf{x}) = \frac{1}{\sqrt{(2\pi)^d |\Sigma|}} \exp\!\left\{-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})\right\}$$

where:
- $\boldsymbol{\mu} \in \mathbb{R}^d$ is the **mean vector** (best estimate)
- $\Sigma \in \mathbb{R}^{d \times d}$ is the **covariance matrix** (uncertainty shape)

In 2D, the iso probability contours are **ellipses**. The covariance matrix $\Sigma$ controls
the size, shape, and orientation of these ellipses.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
sigma_x = 2.0     # std deviation in x   (try 0.5, 1.0, 3.0)
sigma_y = 1.0     # std deviation in y   (try 0.5, 1.0, 3.0)
rho     = 0.0     # correlation coeff.   (try -0.9, 0.0, 0.5, 0.95)
# ─────────────────────────────────────────────────────────────────────────────

rho = np.clip(rho, -0.99, 0.99)

# Three covariance matrices to compare
configs = [
    ('Diagonal (axis aligned)',     np.array([[sigma_x**2, 0], [0, sigma_y**2]])),
    (f'Positive correlation (ρ={rho:.1f})',
         np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                   [rho*sigma_x*sigma_y, sigma_y**2]])),
    (f'Negative correlation (ρ={-abs(rho):.1f})',
         np.array([[sigma_x**2, -abs(rho)*sigma_x*sigma_y],
                   [-abs(rho)*sigma_x*sigma_y, sigma_y**2]])),
]

xg = np.linspace(-6, 6, 200)
yg = np.linspace(-6, 6, 200)
Xg, Yg = np.meshgrid(xg, yg)
pos = np.stack([Xg, Yg], axis=-1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, cov) in zip(axes, configs):
    Z = multivariate_normal(mean=[0, 0], cov=cov).pdf(pos)
    ax.contourf(Xg, Yg, Z, levels=20, cmap='Blues')
    # draw 1 sigma and 2 sigma ellipses
    for ns, clr in [(1, 'tomato'), (2, 'orange')]:
        cov_ellipse(ax, [0, 0], cov, n_std=ns,
                    fill=False, edgecolor=clr, linewidth=2,
                    label=f'{ns}$\\sigma$' if ax is axes[0] else None)
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6); ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(title, fontsize=10)

axes[0].legend(fontsize=9)
plt.suptitle('2D Gaussian contours for three covariance shapes', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

**Things to try:**
1. Set `rho = 0` : the ellipse is axis aligned (x and y are independent).
2. Set `rho = 0.9` : the ellipse tilts at 45 degrees (strong positive correlation).
3. Set `sigma_x = sigma_y` with `rho = 0` : the ellipse becomes a circle.

---

## 6.3 Mean and Covariance Interpretation

The **mean** $\boldsymbol{\mu}$ is the best estimate of the state. The **covariance matrix** $\Sigma$ encodes:
- **Diagonal entries** $\Sigma_{ii} = \sigma_i^2$ : individual uncertainty in each dimension.
- **Off-diagonal entries** $\Sigma_{ij}$ : correlation between dimensions.

$$\Sigma = \begin{bmatrix} \sigma_x^2 & \rho\,\sigma_x\sigma_y \\ \rho\,\sigma_x\sigma_y & \sigma_y^2 \end{bmatrix}$$

**Physical intuition:** if a robot drove diagonally, its x and y errors are correlated.
Knowing that x is larger than expected tells you y is probably larger too ($\rho > 0$).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_pts        = 500     # number of samples   (try 50, 200, 2000)
true_sigma_x = 2.0
true_sigma_y = 1.0
true_rho     = 0.7     # correlation          (try 0.0, 0.7, -0.5)
# ─────────────────────────────────────────────────────────────────────────────

true_cov = np.array([[true_sigma_x**2, true_rho*true_sigma_x*true_sigma_y],
                     [true_rho*true_sigma_x*true_sigma_y, true_sigma_y**2]])
true_mu  = np.array([1.0, 2.0])

rng = np.random.default_rng(7)
samples_corr   = rng.multivariate_normal(true_mu, true_cov, n_pts)
samples_uncorr = rng.multivariate_normal(true_mu,
    np.diag([true_sigma_x**2, true_sigma_y**2]), n_pts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, samp, title in zip(axes,
    [samples_uncorr, samples_corr],
    ['Uncorrelated ($\\rho=0$): circular cloud',
     f'Correlated ($\\rho={true_rho}$): tilted elliptical cloud']):

    ax.scatter(samp[:, 0], samp[:, 1], s=8, alpha=0.5, color='steelblue')
    emp_cov = np.cov(samp.T)
    emp_mu  = samp.mean(axis=0)
    for ns, c in [(1, 'tomato'), (2, 'orange')]:
        cov_ellipse(ax, emp_mu, emp_cov, n_std=ns,
                    fill=False, edgecolor=c, linewidth=2)
    ax.plot(*emp_mu, 'x', color='tomato', markersize=10, markeredgewidth=2)
    ax.set_xlim(-6, 8); ax.set_ylim(-3, 7); ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(title, fontsize=10)

plt.suptitle('Correlation changes the shape of the sample cloud', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

### Sample statistics converge to true parameters

When we compute the **sample mean** $\hat{\mu} = \frac{1}{N}\sum_i x_i$ and the
**sample covariance** $\hat{\Sigma}$ from $N$ samples, these estimates converge to
the true $\mu$ and $\Sigma$ as $N \to \infty$. The next cell shows this convergence.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
true_mu_conv  = 3.0
true_sig_conv = 2.0
max_N         = 5000
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(99)
all_samples = rng.normal(true_mu_conv, true_sig_conv, size=max_N)

Ns = np.arange(10, max_N + 1, 10)
running_mean = np.array([all_samples[:n].mean() for n in Ns])
running_std  = np.array([all_samples[:n].std(ddof=1) for n in Ns])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(Ns, running_mean, color='steelblue', linewidth=1.5)
axes[0].axhline(true_mu_conv, color='tomato', linestyle='--', label=f'True $\\mu$={true_mu_conv}')
axes[0].set_xlabel('Number of samples N'); axes[0].set_ylabel('Sample mean')
axes[0].set_title('Convergence of sample mean'); axes[0].legend()

axes[1].plot(Ns, running_std, color='steelblue', linewidth=1.5)
axes[1].axhline(true_sig_conv, color='tomato', linestyle='--', label=f'True $\\sigma$={true_sig_conv}')
axes[1].set_xlabel('Number of samples N'); axes[1].set_ylabel('Sample std')
axes[1].set_title('Convergence of sample std'); axes[1].legend()

plt.tight_layout(); plt.show()

**Key observation:** With just a few hundred samples, both the mean and standard deviation
are already close to their true values. This is why Monte Carlo methods work in practice:
moderate sample sizes are sufficient for good approximations.

---

## 6.4 Sampling Methods

Sampling is the computational backbone of many robotics algorithms. This section covers
four fundamental methods, from simple to sophisticated.

### 6.4.1 Direct Sampling: the Box Muller Transform

How does a computer generate Gaussian random numbers when all it has is a uniform random
number generator? One classical answer is the **Box Muller transform**:

Given $U_1, U_2 \sim \text{Uniform}(0, 1)$, compute:

$$Z_0 = \sqrt{-2 \ln U_1} \cos(2\pi U_2), \qquad Z_1 = \sqrt{-2 \ln U_1} \sin(2\pi U_2)$$

Then $Z_0$ and $Z_1$ are independent standard Gaussians $\mathcal{N}(0, 1)$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_bm = 5000    # number of samples for Box-Muller (try 500, 2000, 10000)
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(0)
U1 = rng.uniform(0, 1, n_bm)
U2 = rng.uniform(0, 1, n_bm)

# Box-Muller transform
Z0 = np.sqrt(-2 * np.log(U1)) * np.cos(2 * np.pi * U2)
Z1 = np.sqrt(-2 * np.log(U1)) * np.sin(2 * np.pi * U2)

# Compare with numpy's built-in
Z_numpy = rng.normal(0, 1, n_bm)

x = np.linspace(-4, 4, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(Z0, bins=50, density=True, alpha=0.5, color='steelblue',
             edgecolor='white', label='Box Muller samples')
axes[0].plot(x, norm.pdf(x), 'tomato', linewidth=2, label='True $\\mathcal{N}(0,1)$')
axes[0].set_title('Box Muller Transform'); axes[0].legend(fontsize=9)
axes[0].set_xlabel('z'); axes[0].set_ylabel('Density')

axes[1].hist(Z_numpy, bins=50, density=True, alpha=0.5, color='forestgreen',
             edgecolor='white', label='np.random.normal')
axes[1].plot(x, norm.pdf(x), 'tomato', linewidth=2, label='True $\\mathcal{N}(0,1)$')
axes[1].set_title('NumPy built in sampler'); axes[1].legend(fontsize=9)
axes[1].set_xlabel('z'); axes[1].set_ylabel('Density')

plt.tight_layout(); plt.show()

print(f'Box Muller  :  mean = {Z0.mean():.4f},  std = {Z0.std():.4f}')
print(f'np.random   :  mean = {Z_numpy.mean():.4f},  std = {Z_numpy.std():.4f}')

Both methods produce the same distribution. The Box Muller transform is elegant because
it converts two uniform samples into two Gaussian samples with a simple formula.

### 6.4.2 Cholesky Sampling (for Multivariate Gaussians)

To sample from a **multivariate** Gaussian $\mathcal{N}(\boldsymbol{\mu}, \Sigma)$, we need
to produce correlated samples. The key technique is the **Cholesky decomposition**:

1. Decompose $\Sigma = L L^\top$ where $L$ is a lower triangular matrix.
2. Draw $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, I)$ (independent standard normals).
3. Compute $\mathbf{x} = \boldsymbol{\mu} + L\,\mathbf{z}$.

Then $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)$.

**Why it works:** The covariance of $L\mathbf{z}$ is $L \,\text{Cov}(\mathbf{z})\, L^\top = L I L^\top = L L^\top = \Sigma$. The Cholesky factor $L$ is essentially the "square root" of the covariance matrix. It transforms independent samples into correlated ones.

This is **exactly** what `np.random.multivariate_normal` does under the hood.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
chol_mu      = np.array([1.0, 2.0])     # mean vector
chol_cov_xx  = 4.0    # variance in x                   (try 1, 4, 9)
chol_cov_yy  = 1.0    # variance in y                   (try 0.5, 1, 4)
chol_cov_xy  = 1.5    # covariance xy  (must satisfy xy^2 < xx*yy)
n_chol       = 800    # number of samples
# ─────────────────────────────────────────────────────────────────────────────

Sigma_chol = np.array([[chol_cov_xx, chol_cov_xy],
                       [chol_cov_xy, chol_cov_yy]])

# Step 1: Cholesky decomposition
L = np.linalg.cholesky(Sigma_chol)

# Step 2: draw standard normals
rng = np.random.default_rng(42)
z = rng.standard_normal((2, n_chol))          # shape (2, N)

# Step 3: transform
x_samples = chol_mu[:, None] + L @ z           # shape (2, N)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: standard samples z (circular)
axes[0].scatter(z[0], z[1], s=5, alpha=0.4, color='steelblue')
cov_ellipse(axes[0], [0, 0], np.eye(2), n_std=2,
            fill=False, edgecolor='tomato', linewidth=2)
axes[0].set_xlim(-5, 5); axes[0].set_ylim(-5, 5); axes[0].set_aspect('equal')
axes[0].set_title('Step 2: Standard samples $\\mathbf{z}$\n(circular, independent)')
axes[0].set_xlabel('$z_1$'); axes[0].set_ylabel('$z_2$')

# Panel 2: Cholesky factor L
axes[1].axis('off')
axes[1].text(0.5, 0.65,
    '$\\Sigma = L\\, L^\\top$',
    transform=axes[1].transAxes, fontsize=18, ha='center',
    bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.9))
axes[1].text(0.5, 0.40,
    f'L = [{L[0,0]:.3f}  {L[0,1]:.3f}]\n'
    f'    [{L[1,0]:.3f}  {L[1,1]:.3f}]',
    transform=axes[1].transAxes, fontsize=14, ha='center',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='#fff3e0', alpha=0.9))
axes[1].text(0.5, 0.18,
    '$\\mathbf{x} = \\mu + L\\, \\mathbf{z}$',
    transform=axes[1].transAxes, fontsize=16, ha='center')
axes[1].set_title('Step 1: Cholesky factor')

# Panel 3: transformed samples x (elliptical)
axes[2].scatter(x_samples[0], x_samples[1], s=5, alpha=0.4, color='forestgreen')
for ns, clr in [(1, 'tomato'), (2, 'orange')]:
    cov_ellipse(axes[2], chol_mu, Sigma_chol, n_std=ns,
                fill=False, edgecolor=clr, linewidth=2)
axes[2].plot(*chol_mu, 'x', color='tomato', markersize=12, markeredgewidth=2)
axes[2].set_xlim(-5, 8); axes[2].set_ylim(-3, 7); axes[2].set_aspect('equal')
axes[2].set_title('Step 3: Transformed samples $\\mathbf{x}$\n(elliptical, correlated)')
axes[2].set_xlabel('$x_1$'); axes[2].set_ylabel('$x_2$')

plt.suptitle('Cholesky sampling: independent $\\to$ correlated', fontsize=13, y=1.03)
plt.tight_layout(); plt.show()

**Key observation:** The Cholesky factor $L$ acts as a "stretching and rotating" matrix that
transforms a circular cloud of independent samples into an elliptical cloud of correlated
samples. This is the same transformation that appears in the Kalman filter prediction step
when propagating uncertainty.

### 6.4.3 Rejection Sampling

What if we want to sample from a distribution $p(x)$ that is **not** Gaussian? For example,
a **bimodal** distribution (a mixture of two Gaussians) cannot be drawn from directly.

**Rejection sampling** uses a simpler **proposal** distribution $q(x)$ that envelopes $p(x)$:

1. Find a constant $M$ such that $p(x) \leq M \cdot q(x)$ for all $x$.
2. Draw a sample $x^* \sim q(x)$.
3. Draw $u \sim \text{Uniform}(0, 1)$.
4. **Accept** $x^*$ if $u < \frac{p(x^*)}{M \cdot q(x^*)}$; otherwise **reject** it.

The accepted samples follow the distribution $p(x)$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
mix_mu1    = -2.0     # center of first Gaussian mode
mix_mu2    =  3.0     # center of second Gaussian mode
mix_sigma  =  0.8     # shared std dev of both modes
mix_w1     =  0.4     # weight of first mode (mix_w2 = 1 - mix_w1)
proposal_lo = -6.0    # proposal uniform lower bound
proposal_hi =  7.0    # proposal uniform upper bound
n_proposal  = 10000   # total proposals to draw
# ─────────────────────────────────────────────────────────────────────────────

mix_w2 = 1.0 - mix_w1

def target_pdf(x):
    """Bimodal Gaussian mixture."""
    return mix_w1 * norm.pdf(x, mix_mu1, mix_sigma) + mix_w2 * norm.pdf(x, mix_mu2, mix_sigma)

# Uniform proposal density
q_density = 1.0 / (proposal_hi - proposal_lo)

# Find M: upper bound for p(x)/q(x)
x_grid = np.linspace(proposal_lo, proposal_hi, 5000)
M = (target_pdf(x_grid) / q_density).max() * 1.05   # small safety margin

rng = np.random.default_rng(17)
proposals = rng.uniform(proposal_lo, proposal_hi, n_proposal)
u_vals    = rng.uniform(0, 1, n_proposal)

accept_prob = target_pdf(proposals) / (M * q_density)
accepted    = proposals[u_vals < accept_prob]
rejected    = proposals[u_vals >= accept_prob]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(accepted, bins=60, density=True, alpha=0.5, color='forestgreen',
        edgecolor='white', label=f'Accepted ({len(accepted)} samples)')
ax.plot(x_grid, target_pdf(x_grid), 'tomato', linewidth=2.5, label='Target $p(x)$ (bimodal)')
ax.axhline(M * q_density, color='orange', linestyle='--', linewidth=1.5,
           label=f'Envelope $M \\cdot q(x) = {M*q_density:.3f}$')

# Show a few rejected points
n_show = min(300, len(rejected))
ax.scatter(rejected[:n_show], rng.uniform(0, M*q_density*0.95, n_show),
           s=3, alpha=0.2, color='gray', label='Rejected (subset)')

ax.set_xlabel('x'); ax.set_ylabel('Density')
rate = len(accepted) / n_proposal * 100
ax.set_title(f'Rejection Sampling   (acceptance rate: {rate:.1f}%)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'Proposed: {n_proposal},  Accepted: {len(accepted)},  Rate: {rate:.1f}%')

**Key observations:**
- The histogram of accepted samples matches the bimodal target PDF.
- The acceptance rate depends on how well the proposal covers the target. A tight proposal gives a higher rate.
- **In high dimensions, rejection sampling becomes extremely inefficient** because the ratio of "useful" volume shrinks exponentially. This motivates more sophisticated methods like MCMC and importance sampling.

### 6.4.4 Importance Sampling

**Importance sampling** avoids the waste of rejection sampling by keeping every sample but
weighting them:

$$\mathbb{E}_{p}[f(x)] \;=\; \int f(x)\, p(x)\, dx
    \;=\; \int f(x)\, \frac{p(x)}{q(x)}\, q(x)\, dx
    \;\approx\; \frac{1}{N} \sum_{i=1}^N f(x_i)\, w_i$$

where $x_i \sim q(x)$ and the **importance weight** is $w_i = \frac{p(x_i)}{q(x_i)}$.

This is the mathematical basis of the **particle filter** (Chapter 18), where each
particle carries a weight proportional to the likelihood.

Below, we estimate the mean of a target distribution using importance sampling with a
Gaussian proposal. We also show what happens when the proposal is poorly chosen.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
target_mu     = 2.0     # mean of target distribution
target_sigma  = 1.0     # std of target distribution
good_q_mu     = 2.0     # mean of good proposal      (close to target)
bad_q_mu      = 6.0     # mean of bad proposal        (far from target)
q_sigma       = 2.0     # std of both proposals
n_is          = 2000    # number of importance samples
# ─────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(55)

def is_estimate(q_mu, q_sig, n):
    """Importance sampling estimate of E_p[x] using Gaussian proposal."""
    samples = rng.normal(q_mu, q_sig, n)
    # importance weights: p(x) / q(x)
    log_w = (norm.logpdf(samples, target_mu, target_sigma)
           - norm.logpdf(samples, q_mu, q_sig))
    w = np.exp(log_w)
    w_norm = w / w.sum()           # self-normalized weights
    est_mean = np.sum(w_norm * samples)
    ess = 1.0 / np.sum(w_norm**2)  # effective sample size
    return samples, w_norm, est_mean, ess

samp_g, w_g, mean_g, ess_g = is_estimate(good_q_mu, q_sigma, n_is)
rng = np.random.default_rng(55)  # reset for fair comparison
# re-seed so the bad proposal gets fresh draws
rng2 = np.random.default_rng(123)

def is_estimate2(q_mu, q_sig, n, r):
    samples = r.normal(q_mu, q_sig, n)
    log_w = (norm.logpdf(samples, target_mu, target_sigma)
           - norm.logpdf(samples, q_mu, q_sig))
    w = np.exp(log_w)
    w_norm = w / w.sum()
    est_mean = np.sum(w_norm * samples)
    ess = 1.0 / np.sum(w_norm**2)
    return samples, w_norm, est_mean, ess

samp_b, w_b, mean_b, ess_b = is_estimate2(bad_q_mu, q_sigma, n_is, rng2)

x = np.linspace(-4, 12, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (samp, w, q_mu_val, est, ess, label) in zip(axes, [
    (samp_g, w_g, good_q_mu, mean_g, ess_g, 'Good proposal'),
    (samp_b, w_b, bad_q_mu, mean_b, ess_b, 'Bad proposal')]):

    ax.plot(x, norm.pdf(x, target_mu, target_sigma), 'tomato',
            linewidth=2.5, label='Target $p(x)$')
    ax.plot(x, norm.pdf(x, q_mu_val, q_sigma), 'steelblue',
            linewidth=2, linestyle='--', label=f'Proposal $q(x)$, $\\mu_q$={q_mu_val}')

    # plot weighted samples as stems
    idx = np.argsort(-w)[:50]   # top 50 by weight
    ax.stem(samp[idx], w[idx] * 5, linefmt='forestgreen', markerfmt='.',
            basefmt=' ', label='Top 50 weighted samples')

    ax.axvline(est, color='orange', linewidth=2,
               label=f'IS estimate = {est:.3f}')
    ax.axvline(target_mu, color='tomato', linewidth=1, linestyle=':',
               label=f'True mean = {target_mu:.1f}')

    ax.set_xlabel('x'); ax.set_ylabel('Density / weight')
    ax.set_title(f'{label}  (ESS = {ess:.0f} / {n_is})')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(-4, 12)

plt.tight_layout(); plt.show()

print(f'True mean = {target_mu:.3f}')
print(f'Good proposal: IS estimate = {mean_g:.3f}, ESS = {ess_g:.0f}')
print(f'Bad proposal:  IS estimate = {mean_b:.3f}, ESS = {ess_b:.0f}')

**Key observations:**
- The **good proposal** (centered near the target) gives an accurate estimate with high **effective sample size** (ESS).
- The **bad proposal** (centered far from the target) wastes most samples; only a few receive large weights. The ESS drops dramatically and the estimate becomes unreliable.
- **Effective sample size (ESS)** measures how many "useful" samples we have. An ESS close to $N$ is ideal; a very low ESS signals particle degeneracy, the same problem that plagues particle filters.
- In the particle filter (Chapter 18), resampling is used to combat this degeneracy.

---

## 6.5 Confidence Ellipses

In 2D, the set of points at a constant Mahalanobis distance from the mean forms an **ellipse**.
The eigenvalues of $\Sigma$ give the squared semi axis lengths and the eigenvectors give the
orientation.

For a 2D Gaussian, the fraction of probability mass inside the $k\sigma$ ellipse is:

| Ellipse | Probability contained (2D) |
|---------|---------------------------|
| 1$\sigma$ | 39.3% |
| 2$\sigma$ | 86.5% |
| 3$\sigma$ | 98.9% |

Note: these are **different** from the 1D values (68/95/99.7). In 2D, the 1$\sigma$ ellipse
contains only about 39% of the mass because probability spreads over two dimensions.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
ell_mu      = np.array([0.0, 0.0])
ell_sigma_x = 2.0
ell_sigma_y = 1.0
ell_rho     = 0.6
n_ell       = 5000    # samples to check containment fractions
# ─────────────────────────────────────────────────────────────────────────────

ell_cov = np.array([[ell_sigma_x**2, ell_rho*ell_sigma_x*ell_sigma_y],
                    [ell_rho*ell_sigma_x*ell_sigma_y, ell_sigma_y**2]])

rng = np.random.default_rng(10)
samp = rng.multivariate_normal(ell_mu, ell_cov, n_ell)

# Mahalanobis distance for each sample
diff = samp - ell_mu
inv_cov = np.linalg.inv(ell_cov)
mahal = np.sqrt(np.sum(diff @ inv_cov * diff, axis=1))

fig, ax = plt.subplots(figsize=(8, 7))

colors_ell = ['steelblue', 'tomato', 'orange']
for k, clr in zip([1, 2, 3], colors_ell):
    frac = (mahal <= k).mean() * 100
    cov_ellipse(ax, ell_mu, ell_cov, n_std=k,
                fill=False, edgecolor=clr, linewidth=2.5,
                label=f'{k}$\\sigma$ ellipse: {frac:.1f}% inside')

ax.scatter(samp[:, 0], samp[:, 1], s=4, alpha=0.3, color='steelblue')
ax.plot(*ell_mu, 'x', color='tomato', markersize=12, markeredgewidth=3)
ax.set_xlim(-7, 7); ax.set_ylim(-5, 5); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Confidence ellipses with sample containment fractions')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

for k in [1, 2, 3]:
    frac = (mahal <= k).mean() * 100
    print(f'{k}σ ellipse: {frac:.1f}% of samples inside  (theory: {[39.3, 86.5, 98.9][k-1]}%)')

### Robot trajectory with growing uncertainty

Below, a robot drives along a path. At each timestep, the position uncertainty grows
(prediction) and then shrinks when a measurement arrives (update). This pattern
previews the Kalman filter (Chapter 16).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_steps       = 8        # number of timesteps
step_size     = 1.5      # how far the robot moves each step
process_noise = 0.15     # noise added each step (prediction growth)
meas_noise    = 0.3      # measurement noise (update shrinkage)
do_updates    = True     # set False to see pure dead reckoning growth
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 6))

mu_t = np.array([0.0, 0.0])
Sigma_t = np.diag([0.1, 0.1])

trajectory = [mu_t.copy()]

for t in range(n_steps):
    # Prediction: move + add process noise
    angle = 0.3 * t      # gentle curve
    dx = step_size * np.array([np.cos(angle), np.sin(angle)])
    mu_t = mu_t + dx
    Q = process_noise * np.eye(2)
    Sigma_t = Sigma_t + Q

    # Draw prediction ellipse (larger)
    cov_ellipse(ax, mu_t, Sigma_t, n_std=2,
                fill=True, facecolor='orange', alpha=0.15,
                edgecolor='orange', linewidth=1.5)

    if do_updates and t % 2 == 1:
        # Measurement update: shrink the ellipse
        R = meas_noise * np.eye(2)
        K = Sigma_t @ np.linalg.inv(Sigma_t + R)  # Kalman gain
        z = mu_t + rng.multivariate_normal([0, 0], R)  # noisy meas
        mu_t = mu_t + K @ (z - mu_t)
        Sigma_t = (np.eye(2) - K) @ Sigma_t

        cov_ellipse(ax, mu_t, Sigma_t, n_std=2,
                    fill=True, facecolor='forestgreen', alpha=0.2,
                    edgecolor='forestgreen', linewidth=1.5)

    trajectory.append(mu_t.copy())

traj = np.array(trajectory)
ax.plot(traj[:, 0], traj[:, 1], 'o-', color='steelblue', linewidth=2, markersize=6)
ax.plot(traj[0, 0], traj[0, 1], 's', color='tomato', markersize=12, zorder=5)
ax.annotate('Start', traj[0], textcoords='offset points', xytext=(10, -15),
            fontsize=11, color='tomato', fontweight='bold')

# Legend entries
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], color='steelblue', marker='o', linewidth=2, label='Trajectory'),
    mpatches.Patch(facecolor='orange', alpha=0.3, edgecolor='orange', label='Prediction (2$\\sigma$)'),
]
if do_updates:
    legend_elems.append(
        mpatches.Patch(facecolor='forestgreen', alpha=0.3, edgecolor='forestgreen',
                       label='After update (2$\\sigma$)'))
ax.legend(handles=legend_elems, fontsize=10, loc='upper left')
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
title_suffix = 'with measurements' if do_updates else 'dead reckoning only'
ax.set_title(f'Robot trajectory uncertainty ({title_suffix})')
plt.tight_layout(); plt.show()

**Things to try:**
- Set `do_updates = False` to see the ellipses grow without bound (pure dead reckoning).
- Increase `process_noise` to see faster uncertainty growth.
- Decrease `meas_noise` to see sharper corrections after each measurement.

---

## 6.6 Linear Transformation of Gaussians

One of the most powerful properties of the Gaussian distribution: it is **closed under linear transformation**.

If $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \Sigma)$ and we apply the linear function $\mathbf{y} = A\mathbf{x} + \mathbf{b}$, then:

$$\mathbf{y} \sim \mathcal{N}(A\boldsymbol{\mu} + \mathbf{b},\; A\Sigma A^\top)$$

The result is still Gaussian. We do not need to sample or integrate; the new mean and covariance follow from simple matrix algebra. This is **why the Kalman filter works**: the prediction step is a linear transformation, so the predicted belief remains Gaussian.

Below, we apply rotation, scaling, and a general matrix to a 2D Gaussian and verify the formula.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
angle_deg = 45.0    # rotation angle in degrees   (try 0, 30, 90)
scale_x   = 1.5     # x scaling factor            (try 0.5, 1.0, 2.0)
scale_y   = 0.7     # y scaling factor            (try 0.5, 1.0, 2.0)
# ─────────────────────────────────────────────────────────────────────────────

# Original Gaussian
mu_orig  = np.array([0.0, 0.0])
Sig_orig = np.array([[2.0, 0.5],
                     [0.5, 0.8]])

theta = np.radians(angle_deg)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.diag([scale_x, scale_y])
A = R @ S
b = np.array([1.0, 2.0])   # translation

# Analytical transformation
mu_new  = A @ mu_orig + b
Sig_new = A @ Sig_orig @ A.T

# Verify by sampling
rng = np.random.default_rng(77)
n_verify = 3000
x_samples = rng.multivariate_normal(mu_orig, Sig_orig, n_verify)
y_samples = (A @ x_samples.T).T + b

emp_mu  = y_samples.mean(axis=0)
emp_cov = np.cov(y_samples.T)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: original
axes[0].scatter(x_samples[:, 0], x_samples[:, 1], s=3, alpha=0.2, color='steelblue')
for ns, c in [(1, 'tomato'), (2, 'orange')]:
    cov_ellipse(axes[0], mu_orig, Sig_orig, n_std=ns,
                fill=False, edgecolor=c, linewidth=2)
axes[0].set_title('Original $\\mathcal{N}(\\mu, \\Sigma)$')
axes[0].set_xlim(-6, 6); axes[0].set_ylim(-5, 5); axes[0].set_aspect('equal')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Panel 2: transformed samples
axes[1].scatter(y_samples[:, 0], y_samples[:, 1], s=3, alpha=0.2, color='forestgreen')
for ns, c in [(1, 'tomato'), (2, 'orange')]:
    cov_ellipse(axes[1], mu_new, Sig_new, n_std=ns,
                fill=False, edgecolor=c, linewidth=2, linestyle='-')
axes[1].plot(*mu_new, 'x', color='tomato', markersize=10, markeredgewidth=2)
axes[1].set_title(f'Transformed: rot {angle_deg}°, scale ({scale_x}, {scale_y})')
axes[1].set_xlim(-8, 10); axes[1].set_ylim(-6, 10); axes[1].set_aspect('equal')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

# Panel 3: formula verification
axes[2].axis('off')
axes[2].text(0.05, 0.85, 'Formula verification', fontsize=14, fontweight='bold',
             transform=axes[2].transAxes)
axes[2].text(0.05, 0.72,
    f'Analytical mean:      [{mu_new[0]:.4f}, {mu_new[1]:.4f}]\n'
    f'Empirical mean:       [{emp_mu[0]:.4f}, {emp_mu[1]:.4f}]',
    transform=axes[2].transAxes, fontsize=11, fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='#e8f5e9', alpha=0.9))
axes[2].text(0.05, 0.42,
    f'Analytical cov:\n  [{Sig_new[0,0]:.4f}  {Sig_new[0,1]:.4f}]\n'
    f'  [{Sig_new[1,0]:.4f}  {Sig_new[1,1]:.4f}]\n\n'
    f'Empirical cov:\n  [{emp_cov[0,0]:.4f}  {emp_cov[0,1]:.4f}]\n'
    f'  [{emp_cov[1,0]:.4f}  {emp_cov[1,1]:.4f}]',
    transform=axes[2].transAxes, fontsize=11, fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='#e3f2fd', alpha=0.9))
axes[2].text(0.05, 0.12,
    '$\\mathbf{y} = A\\mathbf{x} + \\mathbf{b}$\n'
    '$\\mathrm{Cov}(\\mathbf{y}) = A\\,\\Sigma\\,A^\\top$',
    transform=axes[2].transAxes, fontsize=14)

plt.tight_layout(); plt.show()

**Key observation:** The analytical formula $A\Sigma A^\top$ matches the empirical covariance
computed from the transformed samples. This is the closure property that makes the Kalman
filter prediction step exact. In the Kalman filter, $A$ is the state transition matrix and
$\Sigma$ is the current state covariance.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# Apply three different transformations and compare
demo_matrices = {
    'Rotation (60°)': np.array([[np.cos(np.pi/3), -np.sin(np.pi/3)],
                                [np.sin(np.pi/3),  np.cos(np.pi/3)]]),
    'Scaling (2x, 0.5x)': np.diag([2.0, 0.5]),
    'Shear': np.array([[1.0, 0.8], [0.0, 1.0]]),
}
# ─────────────────────────────────────────────────────────────────────────────

Sig_demo = np.array([[1.0, 0.3], [0.3, 0.6]])
mu_demo  = np.array([0.0, 0.0])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, A_mat) in zip(axes, demo_matrices.items()):
    Sig_t = A_mat @ Sig_demo @ A_mat.T

    # Original ellipse
    cov_ellipse(ax, mu_demo, Sig_demo, n_std=2,
                fill=True, facecolor='steelblue', alpha=0.2,
                edgecolor='steelblue', linewidth=2, label='Original')
    # Transformed ellipse
    cov_ellipse(ax, A_mat @ mu_demo, Sig_t, n_std=2,
                fill=True, facecolor='tomato', alpha=0.2,
                edgecolor='tomato', linewidth=2, label='Transformed')

    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('How different matrices $A$ transform the covariance ellipse', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

**Things to notice:**
- **Rotation** preserves the ellipse size but rotates its orientation.
- **Scaling** stretches or compresses the ellipse along axes.
- **Shear** both stretches and tilts the ellipse.

All three follow the same formula: $\Sigma_{\text{new}} = A\,\Sigma\,A^\top$.

---

## 6.7 Product of Gaussians (Sensor Fusion Preview)

The **product** of two Gaussians is proportional to another Gaussian. This beautiful
property is the mathematical foundation of the **Kalman filter update step**.

For 1D Gaussians:

$$\mathcal{N}(\mu_1, \sigma_1^2) \cdot \mathcal{N}(\mu_2, \sigma_2^2) \propto \mathcal{N}(\mu_f, \sigma_f^2)$$

where:

$$\mu_f = \frac{\sigma_2^2\,\mu_1 + \sigma_1^2\,\mu_2}{\sigma_1^2 + \sigma_2^2}, \qquad \sigma_f^2 = \frac{\sigma_1^2\,\sigma_2^2}{\sigma_1^2 + \sigma_2^2}$$

Notice two crucial properties:
- The fused mean $\mu_f$ lies **between** $\mu_1$ and $\mu_2$, weighted by the inverse variances. The more certain source "pulls" harder.
- The fused variance $\sigma_f^2$ is **smaller** than either individual variance. Fusion always reduces uncertainty.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
prior_mu    = 5.0     # prior belief mean
prior_sigma = 2.0     # prior belief std dev      (try 0.5, 2, 5)
meas_mu     = 7.0     # measurement mean
meas_sigma  = 1.0     # measurement std dev       (try 0.5, 1, 3)
# ─────────────────────────────────────────────────────────────────────────────

# Fused (posterior) parameters
fused_var   = (prior_sigma**2 * meas_sigma**2) / (prior_sigma**2 + meas_sigma**2)
fused_sigma = np.sqrt(fused_var)
fused_mu    = (meas_sigma**2 * prior_mu + prior_sigma**2 * meas_mu) / (prior_sigma**2 + meas_sigma**2)

x = np.linspace(min(prior_mu, meas_mu) - 4*max(prior_sigma, meas_sigma),
                max(prior_mu, meas_mu) + 4*max(prior_sigma, meas_sigma), 500)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, norm.pdf(x, prior_mu, prior_sigma), color='steelblue', linewidth=2.5,
        label=f'Prior: $\\mu_1={prior_mu}$, $\\sigma_1={prior_sigma}$')
ax.fill_between(x, norm.pdf(x, prior_mu, prior_sigma), alpha=0.1, color='steelblue')

ax.plot(x, norm.pdf(x, meas_mu, meas_sigma), color='orange', linewidth=2.5,
        label=f'Measurement: $\\mu_2={meas_mu}$, $\\sigma_2={meas_sigma}$')
ax.fill_between(x, norm.pdf(x, meas_mu, meas_sigma), alpha=0.1, color='orange')

ax.plot(x, norm.pdf(x, fused_mu, fused_sigma), color='tomato', linewidth=3,
        label=f'Fused: $\\mu_f={fused_mu:.2f}$, $\\sigma_f={fused_sigma:.2f}$')
ax.fill_between(x, norm.pdf(x, fused_mu, fused_sigma), alpha=0.15, color='tomato')

ax.axvline(fused_mu, color='tomato', linestyle=':', linewidth=1.5)

ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title('Product of two Gaussians = Kalman filter update')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

print(f'Prior:       mu = {prior_mu:.2f},  sigma = {prior_sigma:.2f}')
print(f'Measurement: mu = {meas_mu:.2f},  sigma = {meas_sigma:.2f}')
print(f'Fused:       mu = {fused_mu:.2f},  sigma = {fused_sigma:.2f}')
print(f'\nThe fused sigma ({fused_sigma:.2f}) is smaller than both '
      f'prior ({prior_sigma:.2f}) and measurement ({meas_sigma:.2f}).')

**Things to try:**
- Make the measurement very precise (`meas_sigma = 0.3`): the fused belief snaps to the measurement.
- Make the prior very precise (`prior_sigma = 0.3`): the fused belief barely moves from the prior.
- Equal uncertainties: the fused mean is exactly halfway between the two.

This is the core of the Kalman filter update. In Chapter 16, we will see the matrix version of
this formula for multivariate states.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
# How the fused sigma changes as measurement precision varies
fixed_prior_sigma = 2.0
meas_sigma_range  = np.linspace(0.1, 5.0, 200)
# ─────────────────────────────────────────────────────────────────────────────

fused_sigmas = np.sqrt(
    (fixed_prior_sigma**2 * meas_sigma_range**2) /
    (fixed_prior_sigma**2 + meas_sigma_range**2)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(meas_sigma_range, fused_sigmas, color='tomato', linewidth=2.5,
        label='Fused $\\sigma_f$')
ax.axhline(fixed_prior_sigma, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Prior $\\sigma_1$ = {fixed_prior_sigma}')
ax.plot(meas_sigma_range, meas_sigma_range, color='orange', linestyle='--',
        linewidth=1.5, label='Measurement $\\sigma_2$')

ax.set_xlabel('Measurement $\\sigma_2$')
ax.set_ylabel('Fused $\\sigma_f$')
ax.set_title('Fused uncertainty is always smaller than both inputs')
ax.legend(fontsize=10)
ax.set_ylim(0, fixed_prior_sigma * 1.2)
plt.tight_layout(); plt.show()

The fused $\sigma_f$ (red curve) always lies **below** both the prior $\sigma_1$ (blue dashed)
and the measurement $\sigma_2$ (orange dashed). This is the power of sensor fusion:
combining two noisy sources always gives a result that is more certain than either one alone.

---

## Exercises

These exercises build on the concepts in this chapter. Each one asks you to write code
and verify a result.

### Exercise 6.1: Verify the 68/95/99.7 rule by sampling

Generate 100,000 samples from $\mathcal{N}(0, 1)$. Compute the fraction of samples
that fall within $\pm 1\sigma$, $\pm 2\sigma$, and $\pm 3\sigma$. Compare with the
theoretical values (68.27%, 95.45%, 99.73%).

In [ ]:
# Exercise 6.1: your code here
rng = np.random.default_rng(42)
samples_ex1 = rng.normal(0, 1, 100_000)

for k in [1, 2, 3]:
    frac = ...  # compute fraction of samples in [-k, k]
    theory = ...  # use scipy: norm.cdf(k) - norm.cdf(-k)
    # print(f'{k}σ: empirical = {frac*100:.2f}%, theory = {theory*100:.2f}%')

### Exercise 6.2: Manual Cholesky for 2x2 matrices

Implement the Cholesky decomposition for a 2x2 positive definite matrix **without** using
`np.linalg.cholesky`. For a matrix $\Sigma = \begin{bmatrix} a & b \\ b & c \end{bmatrix}$,
the lower triangular factor is:

$$L = \begin{bmatrix} \sqrt{a} & 0 \\ b/\sqrt{a} & \sqrt{c - b^2/a} \end{bmatrix}$$

Implement this, verify $L L^\top = \Sigma$, and use it to generate 500 correlated samples.
Compare with `np.linalg.cholesky`.

In [ ]:
# Exercise 6.2: your code here
def cholesky_2x2(Sigma):
    """Compute the Cholesky factor of a 2x2 positive-definite matrix."""
    a, b = Sigma[0, 0], Sigma[0, 1]
    c = Sigma[1, 1]
    L = np.zeros((2, 2))
    # Fill in L here
    # L[0, 0] = ...
    # L[1, 0] = ...
    # L[1, 1] = ...
    return L

Sigma_test = np.array([[4.0, 1.2], [1.2, 2.0]])
L_manual = cholesky_2x2(Sigma_test)
L_numpy  = np.linalg.cholesky(Sigma_test)

print('Manual L:\n', L_manual)
print('NumPy  L:\n', L_numpy)
print('L @ L.T:\n', L_manual @ L_manual.T)
print('Match:', np.allclose(L_manual, L_numpy))

### Exercise 6.3: Rejection sampling from a semicircular distribution

Sample from the semicircular distribution:

$$p(x) = \frac{2}{\pi R^2} \sqrt{R^2 - x^2}, \quad -R \leq x \leq R$$

Use a uniform proposal on $[-R, R]$. Plot the histogram of accepted samples against the
analytical PDF. Report the acceptance rate.

In [ ]:
# Exercise 6.3: your code here
R = 2.0
n_prop = 20000

def semicircle_pdf(x, R):
    return (2.0 / (np.pi * R**2)) * np.sqrt(np.maximum(R**2 - x**2, 0))

# Draw proposals from Uniform(-R, R)
# Accept/reject based on p(x) / (M * q(x))
# Plot the result
# Your code here ...

### Exercise 6.4: Importance sampling for tail probabilities

Estimate $P(X > 3)$ for $X \sim \mathcal{N}(0, 1)$ using importance sampling.
This is a rare event ($P \approx 0.00135$), so direct sampling is inefficient.

Use a **shifted proposal** $q(x) = \mathcal{N}(3, 1)$ to focus samples in the tail.
Compare the importance sampling estimate with `scipy.stats.norm.sf(3)` (the exact value).

In [ ]:
# Exercise 6.4: your code here
n_is_ex = 10000
rng = np.random.default_rng(42)

# 1. Sample from proposal q(x) = N(3, 1)
# 2. Compute importance weights w_i = p(x_i) / q(x_i)
# 3. Estimate P(X>3) = E_p[1_{x>3}] using weighted samples
# 4. Compare with norm.sf(3)

exact = norm.sf(3)
print(f'Exact P(X > 3) = {exact:.6f}')
# print(f'IS estimate    = {is_est:.6f}')

### Exercise 6.5: Product of two 2D Gaussians

For two 2D Gaussians $\mathcal{N}(\boldsymbol{\mu}_1, \Sigma_1)$ and
$\mathcal{N}(\boldsymbol{\mu}_2, \Sigma_2)$, the product is proportional to
$\mathcal{N}(\boldsymbol{\mu}_f, \Sigma_f)$ where:

$$\Sigma_f = (\Sigma_1^{-1} + \Sigma_2^{-1})^{-1}, \qquad \boldsymbol{\mu}_f = \Sigma_f\,(\Sigma_1^{-1}\boldsymbol{\mu}_1 + \Sigma_2^{-1}\boldsymbol{\mu}_2)$$

Implement this formula. Use:
- $\boldsymbol{\mu}_1 = [1, 2]$, $\Sigma_1 = \begin{bmatrix} 2 & 0.5 \\ 0.5 & 1 \end{bmatrix}$
- $\boldsymbol{\mu}_2 = [3, 1]$, $\Sigma_2 = \begin{bmatrix} 1 & -0.3 \\ -0.3 & 1.5 \end{bmatrix}$

Plot all three ellipses (prior, measurement, fused) and verify the fused ellipse is smaller.

In [ ]:
# Exercise 6.5: your code here
mu1 = np.array([1.0, 2.0])
S1  = np.array([[2.0, 0.5], [0.5, 1.0]])
mu2 = np.array([3.0, 1.0])
S2  = np.array([[1.0, -0.3], [-0.3, 1.5]])

# Compute fused mean and covariance
# Sf = ...
# muf = ...

# Plot the three 2-sigma ellipses
# Your code here ...

### Exercise 6.6 (Capstone): Sequential sensor fusion

A robot has a **prior belief** about its 2D position:

$$\text{Prior: } \mathcal{N}\left(\begin{bmatrix}5\\5\end{bmatrix},\; \begin{bmatrix}2 & 0.5\\0.5 & 1\end{bmatrix}\right)$$

It receives two measurements:

1. **GPS** measurement at $[5.5,\, 4.8]$ with isotropic noise $\sigma = 0.8$ (so $\Sigma_{\text{gps}} = 0.64\, I$).
2. **Range bearing to a known landmark** gives a position estimate of $[5.2,\, 5.3]$ with $\Sigma_{\text{rb}} = \begin{bmatrix}0.5 & 0.1\\0.1 & 0.3\end{bmatrix}$.

Fuse sequentially:
- First fuse prior with GPS using the Gaussian product formula.
- Then fuse the result with the range bearing measurement.

Plot the prior, GPS likelihood, range bearing likelihood, posterior after GPS, and
final posterior. Show the ellipses shrinking at each fusion step.

In [ ]:
# Exercise 6.6 (Capstone): your code here

# Prior
mu_prior = np.array([5.0, 5.0])
S_prior  = np.array([[2.0, 0.5], [0.5, 1.0]])

# GPS measurement
mu_gps = np.array([5.5, 4.8])
S_gps  = 0.64 * np.eye(2)

# Range-bearing measurement
mu_rb = np.array([5.2, 5.3])
S_rb  = np.array([[0.5, 0.1], [0.1, 0.3]])

def gaussian_product_2d(mu1, S1, mu2, S2):
    """Compute the product of two 2D Gaussians."""
    Sf = ...  # fill in
    muf = ... # fill in
    return muf, Sf

# Step 1: fuse prior + GPS
# mu_post1, S_post1 = gaussian_product_2d(mu_prior, S_prior, mu_gps, S_gps)

# Step 2: fuse result + range-bearing
# mu_post2, S_post2 = gaussian_product_2d(mu_post1, S_post1, mu_rb, S_rb)

# Plot all ellipses (2-sigma)
# fig, ax = plt.subplots(figsize=(8, 8))
# Your code here ...

---

## Summary

This chapter covered the Gaussian distribution and sampling methods, the two foundations
that underpin almost every estimation algorithm in robotics.

| Concept | Why it matters |
|---------|---------------|
| **1D Gaussian** | Models scalar sensor noise and position uncertainty |
| **Multivariate Gaussian** | Models joint uncertainty over robot state vectors |
| **Covariance matrix** | Encodes both magnitude and correlation of uncertainties |
| **Cholesky sampling** | The engine inside every Kalman filter implementation |
| **Rejection sampling** | General purpose sampling from arbitrary distributions |
| **Importance sampling** | The mathematical basis of particle filters |
| **Confidence ellipses** | The standard visualization of uncertainty in SLAM |
| **Linear transformation** | Why Kalman filter prediction is exact for linear systems |
| **Gaussian product** | The Kalman filter update step in its simplest form |

**Next chapter:** Chapter 7 takes these tools further and shows how uncertainty propagates
through nonlinear functions, motivating the Extended Kalman Filter (EKF).